# Recommender Systems - Mini Challenge FS26

In this minichallenge we will explore a MovieLens dataset and implement several recommender systems and evaluation methods. Subsequently we will optimize these methods and compare the results. 

**Submission deadline:** Sunday of SW11 at 18:00 

## Guidelines for Implementation and Submission
- Code must be written in Python. The versions of all used packages must be given for reproducability.
- You may respond in English or German.
- We develop numerous algorithms ourselves. Unless explicitly stated otherwise, only the following libraries may be used in Python: numpy, matplotlib, seaborn, pandas. 
- Follow good coding practices and write modular, reusable code.
- The submitted solution must contain all codes and the results. No code may be outsourced.
- All pathes must be relative and just downloading your repo must be executable without modifications.
- Only fully running code is graded. The notebook must run sequential from start to end.
- During development, if computation time is too long for productive prototyping and debugging work, it is recommended to reduce the dataset to a fraction of its original. However, final results must be calculated on the full dataset. 
- All plots must be fully labeled (title, axes, labels, colorbar, etc.) so that the plot can be easily understood.
- Each plot must be accompanied by a brief discussion, which explains the plot and captures the key insights that become visible.
- Only fully labeled plots with an accompanying discussion will be assessed.
- The last commit in your fork of the repo before the submission deadline counts as the submission.
- Points will be deducted if you write inconsise (Denial of service will be punished) or if I read text not written for me but for the user of ChatGPT oir similar. 
- If you would like to submit and have the mini-challenge assessed, please send a short email to the subject expert (moritz.kirschmann@fhnw.ch) within 2 days after submission.
- Please do not delete, duplicate, or move the existing cells. This leads to problems during the correction. However, you may add as many additional cells as you like.

## Exercises

### Exercise 1 - A deep exploration of the dataset (17 points)
We will work with a subset of the MovieLens dataset. This subset is located under ``data/ml-latest-small``. Read the ``README.txt``carefully. 
Open the files. 

a) Describe the available data.

b) Find and fix bad data (e.g. duplicates, missing values, etc.).

Generate lists of

c) - Top 20 movies by average rating

d) - Top 20 movies by number of views

e) What is the range of the ratings? 

f) Which genre has be rated how many times?

g) How sparse is the User Rating Matrix?

Plot the following:

h) How many users have rated how many movies

i) Which rating is given how often over time with a time resolution of month 

j) Which rating is given how often per genre

k) The rating distributions of 10 random movies

l) The rating distributions of 3 movies that you have watched

m) How many users give which average rating

n) How often a movie was rated as a function of average rating

o) A heatmap of the User Item Matrix

p) A heatmap of the User Item Matrix for the 100 most rated movies for the 50 users with most ratings


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import seaborn as sns

**a) Describe the available data**

As the provided summary states the dataset describes 5-star rating and free-text tagging activity from MovieLens, a movie recommendation service. It contains 100836 ratings and 3683 tag applications across 9742 movies made by 610 users between March 29, 1996 and September 24, 2018

In [ ]:
movies = pd.read_csv("./Data/ml-latest-small/movies.csv")
ratings = pd.read_csv("./Data/ml-latest-small/ratings.csv")
tags = pd.read_csv("./Data/ml-latest-small/tags.csv")

movies.info()
ratings.info()
tags.info()

The dataset consists of three main components: movies, ratings, and tags.

The `movies` dataset contains information on 9,742 movies each with unique `movieId`, including their titles and associated genres. Genres are provided as categorical descriptors and may include multiple labels per movie.

The `ratings` dataset contains 100,836 user ratings. Each entry represents a rating given by a specific user (`userId`) to a specific movie (`movieId`). Users can rate multiple movies, resulting in multiple entries per user. Ratings are given on a 5-star scale with half-star increments, ranging from 0.5 to 5.0. Additionally, each rating includes a timestamp indicating when the rating was submitted, measured in seconds since January 1, 1970 (UTC).

The `tags` dataset contains user-generated annotations for movies. Each entry links a user (`userId`) to a movie (`movieId`) and includes a tag, typically a short word or phrase describing some aspect of the movie. Similar to the ratings data, a timestamp is provided to indicate when the tag was created.

Overall, the dataset combines explicit feedback (ratings) with descriptive metadata (genres and tags).

**b) Find and fix bad data (e.g. duplicates, missing values, etc.).**

Beyond duplicates and missing values, I will also check for data consistency, validity, and structural issues.


In [ ]:
# 1) Ensuring correct data types
ratings['userId'] = ratings['userId'].astype(int)
ratings['movieId'] = ratings['movieId'].astype(int)
movies['movieId'] = movies['movieId'].astype(int)
ratings['rating'] = ratings['rating'].astype(float)
tags['userId'] = tags['userId'].astype(int)
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit='s')
tags['timestamp'] = pd.to_datetime(tags['timestamp'], unit='s')

# 2) No duplicates
movies = movies.drop_duplicates(subset=['movieId'])
ratings = ratings.drop_duplicates(subset=['userId', 'movieId'])

# 3) No NaN
def nan_check(data):
    total_nans = data.isna().sum().sum()
    per_column = data.isna().sum()
    any_nans = data.isna().any().any()

    print("Total NaNs:", total_nans)
    print("\nNaNs per column:")
    print(per_column)
    print("\nAny NaNs present:", any_nans)

    return total_nans, per_column, any_nans

for name, data in [("movies", movies), ("ratings", ratings), ("tags", tags)]:
    print(f"\n--- {name} ---")
    nan_check(data)

# 4) No invalid ratings
invalid_ratings = ratings[(~ratings['rating'].between(0.5, 5.0)) | (ratings['rating'] % 0.5 != 0)]
print("Number of invalid ratings:", len(invalid_ratings))

# 5)  movieIds in ratings must be in movies
if ratings['movieId'].isin(movies['movieId']).all():
    print("All movieIds in ratings exist in movies")
else:
    print("There are ratings for movies not present in the movies dataset")


I checked for common data quality issues and cleaned where necessary. I convertent columns to appropriate data types to ensure consistency, and made sure the timestamps are in a readable datetime format. Duplicate entries were then removed from the `movies` and `ratings` datasets based on their identifying columns.

Then I checked all three datasets (`movies`, `ratings`, and `tags`) for missing values with a reusable function. No missing values were found in any column. The ratings were also validated to ensure that they lie within the allowed range from 0.5 to 5.0 and follow the required half-star increments. No invalid ratings were detected. Finally, referential integrity was verified by confirming that all `movieId` values appearing in the `ratings` dataset also exist in the `movies` dataset.

Overall, the dataset seems clean, consistent, and suitable for building recommendation systems as expected from GroupLens.

**c) - Top 20 movies by average rating**

In [ ]:
# Make a dataframe with title and average ratings
movieId_avrating = ratings.groupby('movieId')['rating'].mean().reset_index()
movies_with_ratings = movies.merge(movieId_avrating, on='movieId', how='inner')
print(movies_with_ratings.sort_values(by='rating', ascending=False).head(20)) # print top 20

I made a list (not a literal Python list) of the top 20 movies based on their average rating. It was generated by grouping the ratings by movieId, computing the mean rating, and sorting the results in descending order. It is clear that the top spots are dominated by movies with perfect scores, which likely have only very few reviews. In a recommender system, a more robust approach would be to introduce a damping or weighting mechanism to penalise entries with fewer reviews. However, I assume this is not required for this task and will consider it later when building the actual recommendation system.

**d) - Top 20 movies by number of views**

In [ ]:
# A dataframe with title and amount of ratings
rating_amount = ratings.groupby('movieId')['rating'].count().reset_index(name='num_ratings')
movies_with_rating_amount = movies.merge(rating_amount, on='movieId', how='inner')
print(movies_with_rating_amount.sort_values(by='num_ratings', ascending=False).head(20)) # print top 20

I presumed that “top 20 by views” refers to the top 20 movies by the number of reviews in the ratings dataset. I used the same technique as for the top 20 list by average rating. It becomes clear that the most viewed movies are not the same as the top 20 highest-rated movies. This is likely because a larger number of reviewers introduces more variation in the scores due to users’ individual preferences. This indicates that the number of ratings should be taken into account when designing a robust recommender system.

**e) What is the range of the ratings?**

In [ ]:
print("Min rating:", ratings['rating'].min()) # As expected the worst rating is the lowest possible rating of 0.5
print("Max rating:", ratings['rating'].max()) # As expected the best rating is the highest possible rating of 5.0

**f) Which genre has be rated how many times?**

In [ ]:
# merge ratings with movies to get genres per rating
ratings_with_genres = ratings.merge(movies[['movieId', 'genres']], on='movieId')

# split genres and explode into separate rows
genres_exploded = ratings_with_genres['genres'].str.split('|').explode()

# How often each genre was rated
genre_counts = genres_exploded.value_counts()

print(genre_counts) # Shows how many times has each genre been rated

**g) How sparse is the User Rating Matrix?**

Sparsity describes how many entries in the user–item matrix are missing (i.e., movies that users have not rated). It can be calculated as:

$$ sprasity = 1 - \frac{number~of~ratings}{number~of~users~\times~number~of~movies}$$

In [ ]:
n_users = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
n_ratings = len(ratings)

sparsity = 1 - (n_ratings / (n_users * n_movies))

print("Number of users:", n_users)
print("Number of movies:", n_movies)
print("Number of ratings:", n_ratings)
print("Sparsity:", sparsity)

I calculated the sparsity of the user–rating matrix and found it to be approximately 0.983. This means that about 98.3% of all possible user–movie interactions are missing, and only a very small fraction of ratings is actually observed. This confirms that the dataset is highly sparse, which is typical for recommender systems and makes the task of generating accurate recommendations more challenging.

**h) How many users have rated how many movies**

In [ ]:
# How many movies each user rated
ratings_per_user = ratings.groupby('userId')['movieId'].count()

# plot distribution
plt.figure(figsize=(10, 6))
plt.hist(ratings_per_user, bins=30)

plt.title("Distribution of Number of Ratings per User")
plt.xlabel("Number of Movies Rated")
plt.ylabel("Number of Users")

plt.tight_layout()
plt.show()

As I already intreperted by the calculated sparsity, the left-skewed histogram shows that most users have rated only a small number of movies, while a few users have rated a much larger number. This highlights the uneven distribution of user activity in the dataset.

**i) Which rating is given how often over time with a time resolution of month**

In [ ]:
# timestamp -> datetime
ratings['timestamp'] = pd.to_datetime(ratings['timestamp'], unit='s')

# extract month
ratings['month'] = ratings['timestamp'].dt.month

# ratings per month and rating
month_rating = (
    ratings
    .groupby(['month', 'rating'])
    .size()
    .unstack(fill_value=0)
)

# normalize per month
month_rating_norm = month_rating.div(month_rating.sum(axis=1), axis=0)

# column sort
month_rating_norm = month_rating_norm.sort_index(axis=1)

# rating colors more or less based on color spectrum
rating_colors = {
    0.5: 'purple',
    1.0: 'pink',
    1.5: 'blue',
    2.0: 'turquoise',
    2.5: 'cyan',
    3.0: 'green',
    3.5: 'yellow',
    4.0: 'orange',
    4.5: 'red',
    5.0: 'brown'
}

# fixed colors
colors = [rating_colors[r] for r in month_rating_norm.columns]

# plot
month_rating_norm.plot(kind='bar', stacked=True, figsize=(12, 7), color=colors)

plt.title("Normalized rating distribution per month (All Years Aggregated)")
plt.xlabel("Month")
plt.ylabel("Proportion of Ratings")

plt.xticks(
    ticks=range(12),
    labels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
            'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
    rotation=0)

plt.legend(title="Rating", loc='center left', bbox_to_anchor=(1, 0.5))
plt.tight_layout()
plt.show()

My key takeaway from this plot is that higher ratings (around 4.0 – 5.0) make the majority of ratings, while lower ratings tend to get rarer the lower they are. The time of the year does not really seem to have a big impact on the user's rating score. This indicates a positive bias in the dataset, where users are more likely to rate movies they enjoyed.

**j) Which rating is given how often per genre**

In [ ]:
# merge and explode genres
ratings_genres = ratings.merge(movies[['movieId', 'genres']], on='movieId')
ratings_genres = ratings_genres.assign(
    genre=ratings_genres['genres'].str.split('|')
).explode('genre')


ratings_genres = ratings_genres[ratings_genres['genre'] != '(no genres listed)'] # remove placeholer

# count ratings per genre
genre_rating = (
    ratings_genres
    .groupby(['genre', 'rating'])
    .size()
    .unstack(fill_value=0)
)

# normalize for better visualization
genre_rating_norm = genre_rating.div(genre_rating.sum(axis=1), axis=0)

# same color sheme as bafore
rating_colors = {
    0.5: 'purple',
    1.0: 'pink',
    1.5: 'blue',
    2.0: 'turquoise',
    2.5: 'cyan',
    3.0: 'green',
    3.5: 'yellow',
    4.0: 'orange',
    4.5: 'red',
    5.0: 'brown'}

# colors in correct order
colors = [rating_colors[r] for r in genre_rating_norm.columns]

# plot
genre_rating_norm.plot(kind='bar', stacked=True, figsize=(14, 8), color=colors)

plt.title("Normalized Rating Distribution per Genre")
plt.xlabel("Genre")
plt.ylabel("Proportion of Ratings")

plt.xticks(rotation=45)
plt.legend(title="Rating", loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()

As already observed before, higher ratings occur more frequently than lower ones. This pattern is consistent across all genres, where ratings around 4.0–5.0 dominate, while very low ratings are relatively rare. This suggests a general positive bias in user ratings, independent of month and genre.

**k) The rating distributions of 10 random movies**

In [ ]:
# 10 random movies each run
random_movies = movies.sample(n=10, random_state=None)  # change random_state to int like 1 ro remove randomness

# merge with ratings
ratings_subset = ratings.merge(random_movies[['movieId', 'title']], on='movieId')

# plot distributions
plt.figure(figsize=(12, 6))

for title, group in ratings_subset.groupby('title'):
    counts = group['rating'].value_counts().sort_index()
    plt.plot(counts.index, counts.values, marker='o', label=title)

plt.title("Rating Distributions of 10 Random Movies")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")
plt.gca().yaxis.set_major_locator(MaxNLocator(integer=True))

plt.legend(bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show() # Cool plot, not much to say

**l) The rating distributions of 3 movies that you have watched**

In [ ]:
# movie selection (partial match to be safe)
selected_movies = movies[movies['title'].str.contains('Toy Story', case=False)] # This would not have worked if Toy Story 4 was in the data

# merge with ratings
ratings_subset = ratings.merge(selected_movies[['movieId', 'title']], on='movieId')

# plot
plt.figure(figsize=(10, 6))

for title, group in ratings_subset.groupby('title'):
    counts = group['rating'].value_counts().sort_index()
    plt.plot(counts.index, counts.values, marker='o', label=title)

plt.title("Rating distributions of selected movies")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")

plt.legend()
plt.tight_layout()
plt.show()

I consider the Toy Story trilogy to be one of the most significant animated film series. The rating distributions reflect this, as all three movies receive predominantly positive reviews, with ratings concentrated around higher values. However, the number of ratings decreases for the later installments, suggesting that fewer users engaged with the newer movies. In my personal opinion, the movies improved with each sequel, culminating in Toy Story 3 as a strong conclusion to the series. Toy Story 4 is not included in the dataset, but I found it to be a disappointing addition. It felt unnecessary and, in my view, did not contribute meaningfully beyond commercial reasons.

**m) How many users give which average rating**

In [ ]:
# average rating per user
user_avg = ratings.groupby('userId')['rating'].mean()

# plot
plt.figure(figsize=(10, 6))
plt.hist(user_avg, bins=20)

plt.title("Distribution of Users' Average Ratings")
plt.xlabel("Average Rating per User")
plt.ylabel("Number of Users")

plt.tight_layout()
plt.show()

I calculated the average rating for each user and visualised the distribution using a histogram. As observed earlier, most users tend to give relatively high ratings, with the majority of average ratings lying above approximately 3.25. This confirms that users generally rate movies positively, leading to a noticeable skew towards higher average ratings.

**n) How often a movie was rated as a function of average rating**

In [ ]:
# average rating and number of ratings per movie
movie_stats = ratings.groupby('movieId')['rating'].agg(['mean', 'count']).reset_index()

# plot
plt.figure(figsize=(10, 6))
plt.scatter(movie_stats['mean'], movie_stats['count'], alpha=0.5)

plt.title("Number of Ratings vs Average Rating per Movie")
plt.xlabel("Average Rating")
plt.ylabel("Number of Ratings")

plt.tight_layout()
plt.show()

The scatter plot shows the relationship between the average rating of a movie and how often it was rated. Most movies have relatively few ratings and are spread across a wide range of average scores. However, movies with a large number of ratings tend to cluster around average ratings between roughly 3.0 and 4.5. Very highly rated movies (close to 5.0) usually have only a small number of ratings, which suggests that these scores are less reliable. I had not quite noticed this before, as I was mainly looking at how often a rating of 5.0 occurred within a genre or month, rather than considering how many total ratings those movies had. Overall, this indicates that popular movies tend to receive more moderate but stable ratings, while extreme ratings are often based on fewer user opinions.

**o) A heatmap of the User Item Matrix**

**p) A heatmap of the User Item Matrix for the 100 most rated movies for the 50 users with most ratings**

In [ ]:
# o
# create user matrix
user_item = ratings.pivot(index='userId', columns='movieId', values='rating')

user_item_sample = user_item.sample(n=100, random_state=None) # sampling is necessary for visibility

# plot heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(user_item_sample, cmap='viridis', cbar=True)

plt.title("User–Item Matrix (Sample of Users)")
plt.xlabel("Movie ID")
plt.ylabel("User ID")

plt.tight_layout()
plt.show()

In [ ]:
# p
# top 50 users by number of ratings
top_users = ratings['userId'].value_counts().head(50).index

# top 100 movies by number of ratings
top_movies = ratings['movieId'].value_counts().head(100).index

# filter data
filtered = ratings[
    (ratings['userId'].isin(top_users)) &
    (ratings['movieId'].isin(top_movies))
]

# matrix
user_item_filtered = filtered.pivot(index='userId', columns='movieId', values='rating')

# plot heatmap
plt.figure(figsize=(14, 8))
sns.heatmap(user_item_filtered, cmap='viridis', cbar=True)

plt.title("User–Item Matrix (Top 50 users & top 100 movies)")
plt.xlabel("Movie ID")
plt.ylabel("User ID")

plt.tight_layout()
plt.show()

The full user–item matrix appears very empty, even after sampling, which is due to the high sparsity of the dataset we calculated in a previous cell. Since most users have only rated a small subset of all available movies, the majority of entries in the matrix are missing. In contrast, the heatmap for the most active users and most frequently rated movies appears much denser and reveals clearer patterns. This demonstrates how sparsity strongly affects the visibility of structure in the data.

### Exercise 2 - Building a baseline RS (7 points)
In this exercise we will build a baseline RS and functions to calculate fundamental performance metrics. 

Build the following baseline RS to predict Top-N (default N=20):
1. In reference to the book *Collaborative Filtering Recommender Systems by Michael D. Ekstrand, John T. Riedl and Joseph A. Konstan* (p. 91ff) implement the baseline predictor $$ b_{u,i}= \mu +b_u +b_i $$

Elements:

* $b_{u,i}$: predicted rating for user $u$ and movie $i$
* $\mu$: global average rating across all ratings
* $b_u$: user bias / user offset
  * captures whether user $u$ generally rates higher or lower than average
* $b_i$: item bias / movie offset
  * captures whether movie $i$ is generally rated higher or lower than expected



Regularized user offset: $$ b_u = \frac{1}{|I_u| + \beta_u} \sum_{i \in I_u} (r_{u,i} - \mu) $$
* $r_{u,i}$: actual rating given by user $u$ to movie $i$
* $I_u$: set of movies rated by user $u$
* $|I_u|$: number of movies rated by user $u$
* $\beta_u$: damping factor for user bias
    * reduces extreme user offsets when a user has few ratings

Regularized item average offset: $$ b_i = \frac{1}{|U_i| + \beta_i} \sum_{u \in U_i} (r_{u,i} - b_u - \mu) . $$
* $U_i$: set of users who rated movie $i$
* $|U_i|$: number of users who rated movie $i$
* $\beta_i$: damping factor for item bias
    * reduces extreme movie offsets when a movie has few ratings

Build a recommender system upon this baseline predictor. Set the default damping factors $\beta_u$ and $\beta_i$ both to 20.

In simple terms:

* $\mu$ gives the general rating level.
* $b_u$ adjusts for users who rate stricter or more generously.
* $b_i$ adjusts for movies that are generally liked or disliked.
* The damping factors prevent unreliable offsets from users or movies with very few ratings.

2. Build a RS that recommends based on *random* recommendations.  

Output the recommendations for three example users (Ids 1, 3 and 7) and the default parameters. Give the titles of the recommended movies and their predicted scores not just their Ids.

In [ ]:
# first formula: b_u = 1/(|I_u| + beta_u) * sum(r_ui - mu)
def compute_user_bias(ratings, mu, beta_u=20):
    # number of ratings per user |I_u|
    user_counts = ratings.groupby('userId').size()

    # sum of (r_ui - mu) per user
    user_deviation = ratings.groupby('userId')['rating'].apply(lambda x: (x - mu).sum())

    # formula in code
    b_u = user_deviation / (user_counts + beta_u)

    return b_u

# second formula: b_i = 1 / (|U_i| + beta_i) * sum(r_ui - b_u - mu)
def compute_item_bias(ratings, mu, b_u, beta_i=20):
    # map user bias to each rating
    ratings_with_bu = ratings.copy()
    ratings_with_bu['b_u'] = ratings_with_bu['userId'].map(b_u)

    # number of ratings per item |U_i|
    item_counts = ratings_with_bu.groupby('movieId').size()

    # sum of (r_ui - b_u - mu) per item
    item_deviation = ratings_with_bu.groupby('movieId').apply(
        lambda x: (x['rating'] - x['b_u'] - mu).sum()
    )

    # formula in code
    b_i = item_deviation / (item_counts + beta_i)

    return b_i


# third formula: mu + b_u + b_i
def predict_baseline(user_id, movie_id, mu, b_u, b_i):
    # get user bias (default 0 if user not seen)
    bu = b_u.get(user_id, 0)

    # get item bias (default 0 if movie not seen)
    bi = b_i.get(movie_id, 0)

    # baseline prediction rating
    return mu + bu + bi

# THE BASELINE PREDICTOR
def recommend_baseline(user_id, ratings, movies, beta_u=20, beta_i=20, n=20):
    mu = ratings['rating'].mean()
    b_u = compute_user_bias(ratings, mu, beta_u)
    b_i = compute_item_bias(ratings, mu, b_u, beta_i)

    rated_movies = ratings.loc[ratings['userId'] == user_id, 'movieId']
    candidates = movies[~movies['movieId'].isin(rated_movies)].copy()

    candidates['predicted_score'] = candidates['movieId'].apply(
        lambda movie_id: predict_baseline(user_id, movie_id, mu, b_u, b_i)
    )

    recs = (
        candidates
        .sort_values(by='predicted_score', ascending=False)
        .head(n)[['title', 'predicted_score']]
        .copy()
    )

    recs.insert(0, "Rank", range(1, len(recs) + 1))
    recs['predicted_score'] = recs['predicted_score'].round(3) # more exact predicted score
    # recs['predicted_score'] = ((recs['predicted_score'] * 2).round() / 2).map(lambda x: f"{x:.1f}")  # rounds to MovieLens 0.5 rating steps

    return recs

# THE RANDOM PREDICTOR
def recommend_random(user_id, ratings, movies, n=20):
    # movies the user has already rated
    rated_movies = ratings.loc[ratings['userId'] == user_id, 'movieId']

    # candidate movies = movies not yet rated by the user
    candidates = movies[~movies['movieId'].isin(rated_movies)].copy()

    # randomly select N movies
    recommendations = (
        candidates
        .sample(n=n, random_state=None)
        [['movieId', 'title']]
    )

    return recommendations

In [ ]:
print("Recommendations for user 1:",
      recommend_baseline(1, ratings, movies),
      recommend_random(1, ratings, movies))

print("Recommendations for user 3:",
      recommend_baseline(3, ratings, movies),
      recommend_random(3, ratings, movies))

print("Recommendations for user 7:",recommend_baseline(7, ratings, movies),
      recommend_random(7, ratings, movies))

### Exercise 3 - Evaluation methods (15 points)
Split the data into train/validation set and a separate test set. This test set shall contain the first 20% of the users and shall not be used at all before exercise 10. With the remaining 80% do the following: 
Implement a function to partition your dataset for an offline evaluation based on holding out of random users with 5x cross validation with a 80/20 train/validation split. Within the validation set implement a masking with *all but n* approach. 
See page 2942 of https://jmlr.csail.mit.edu/papers/volume10/gunawardana09a/gunawardana09a.pdf for details on this approach. 

Choose the number of masked items n reasonably and explain your considerations.

Implement functions to calculate the following metrics:
- *Mean Absolute Error (MAE)* 
- *Root Mean Square Error (RMSE)*
- *Precision@N* with default $N=5$ and relevance threshold 4.0 stars.
- *Recall@N* with default $N=5$ and relevance threshold 4.0 stars.
- *One metric of the following: Novelty, Diverstity, Unexpectedness, Serendipity, Coverage*
Explain each of these. How does the relevance threshold influence the metrics? How would you choose this parameter?

Note: For *precision@N* and *Recall@N* use the definitions from https://medium.com/@m_n_malaeb/recall-and-precision-at-k-for-recommender-systems-618483226c54 with one exception: In case of the denominator being zero, set the metric to 0. 

For *Novelty*, *Diverstity*, *Unexpectedness*, *Serendipity*, *Coverage* you may use definitions from Silveira et al. https://link.springer.com/article/10.1007/s13042-017-0762-9

**Novelty** is how unlikely it is that the user already knows the item.
At the system level, it typically means Novelty ensures that recommended items are new to the user, rather than items they have already interacted with.

$$R_u \cap I_u = \emptyset$$

- $I$: all items in the system
- $I$: list of user's items
- $R_u$: list of recommended items to the user

As a metric of a recommendation list, the formula is: $ nov_u = (1 / |R_u|) Σ_{i ∈ R_u} -log(popularity(i)) $

- $popularity(i)$ = number of ratings for item i

**Diversity** is a concept concerned with the diversity of items in the recommendation list. It is typically computed using pairwise similarity (e.g., cosine similarity), where lower similarity between items indicates higher diversity.

- Cosine Similarity: $ sim = \frac{i * j}{||i||*||j||}  $
- Diversity: $ div = 1 - \frac{i * j}{||i||*||j||}  $

**Unexpectedness** in recommendation systems refers to how much the recommended items deviate from what the user expects, introducing an element of surprise by avoiding obvious or predictable suggestions.

**Serendipity** in recommendation systems refers to delivering useful and relevant items that are both unexpected and pleasantly surprising to the user. $ ser_u = \frac{|(R_u \ E_u) \cap Rel_u |}{R_u} $

- $E_u$: expected items
- $Rel_u$: relevant items (true rating > 4.0)

**Coverage** measures the proportion of items in the catalog for which the recommender system is able to generate predictions.
$ cov = \frac{|I_p|}{I} $

- $I_p$: items for which the system can make predictions


Serendipity is the metric most directly influenced by a relevance threshold, as it explicitly depends on distinguishing relevant from non-relevant items. In contrast, the other metrics listed in the article are largely independent of the rating threshold. It is conceptually appealing, since it captures multiple aspects of recommendation quality, including surprise and relevance, while avoiding the ambiguity of unexpectedness as a standalone metric. Although novelty and diversity are also important—ensuring that recommendations are new to the user and sufficiently varied.

In [ ]:
# 1) Separate test set: 20% of users
def create_user_test_split(ratings, test_size=0.2, random_state=42, keep_users=[3, 5, 7]):
    unique_users = np.sort(ratings['userId'].unique())

    # users that are allowed to go into the test set
    candidate_test_users = np.array([
        user for user in unique_users
        if user not in keep_users # Without this some tests later on, become impossible
    ])

    np.random.seed(random_state)
    shuffled_users = np.random.permutation(candidate_test_users)

    n_test_users = int(len(unique_users) * test_size)

    test_users = shuffled_users[:n_test_users]

    remaining_users = np.array([
        user for user in unique_users
        if user not in test_users
    ])

    test_set = ratings[ratings['userId'].isin(test_users)].copy()
    remaining_set = ratings[ratings['userId'].isin(remaining_users)].copy()

    return remaining_set, test_set, remaining_users, test_users


# 2) 5-fold cross validation on remaining 80% of users
def create_user_cv_splits(remaining_set, remaining_users, n_folds=5, random_state=42):
    np.random.seed(random_state)

    users = np.array(remaining_users)
    shuffled_users = np.random.permutation(users)

     # split users into 5 equal parts -> each part ca. 20% of users
    user_folds = np.array_split(shuffled_users, n_folds)

    folds = []

    for fold in range(n_folds):
        # one of the 5 parts as validation -> ca. 20% of users
        validation_users = user_folds[fold]

        # all remaining 4 parts combined -> ca. 80% of users
        train_users = np.concatenate(
            [user_folds[i] for i in range(n_folds) if i != fold]
        )

        train_set = remaining_set[remaining_set['userId'].isin(train_users)].copy()
        validation_set = remaining_set[remaining_set['userId'].isin(validation_users)].copy()

        folds.append({
            'fold': fold + 1,
            'train_set': train_set,
            'validation_set': validation_set,
            'train_users': train_users,
            'validation_users': validation_users
        })

    return folds

# 3) separate 20% test set
remaining_set, test_set, remaining_users, test_users = create_user_test_split(ratings)

# 4) create 5-fold cross validation splits from the remaining 80%
folds = create_user_cv_splits(remaining_set, remaining_users)

# 5) results
print("Remaining set ratings:", len(remaining_set))
print("Test set ratings:", len(test_set))
print("Remaining users:", len(remaining_users))
print("Test users:", len(test_users))

for fold in folds:
    print(f"\nFold {fold['fold']}")
    print("Train users:", len(fold['train_users']))
    print("Validation users:", len(fold['validation_users']))
    print("Train ratings:", len(fold['train_set']))
    print("Validation ratings:", len(fold['validation_set']))

*All but $n$ approach* is an important offline evaluation method used to filter for the most promising recommendation algorithms, so only
the best go through the expensive online testing. It means *keeping n known items per user and hiding the rest*. A small n of 5 - 10 simulates the *cold-start* problem, whereas a higher amount makes for a more stable evaluation.

I will pick $n$ = 11 as a balance between realism and evaluation stability, but will experiment with other values afterward.

In [ ]:
# all-but-n approach: keep n ratings visible, hide the rest
def create_all_but_n_split(ratings, validation_users, n=11, random_state=42):
    validation_mask = ratings["userId"].isin(validation_users)

    validation_ratings = ratings[validation_mask]
    non_validation_ratings = ratings[~validation_mask]

    hidden_parts = []
    visible_parts = []

    for user_id, user_ratings in validation_ratings.groupby("userId"):
        if len(user_ratings) <= n:
            visible = user_ratings
            hidden = user_ratings.iloc[0:0]
        else:
            visible = user_ratings.sample(n=n, random_state=random_state)
            hidden = user_ratings.drop(visible.index)

        visible_parts.append(visible)
        hidden_parts.append(hidden)

    validation_visible = pd.concat(visible_parts)
    validation_hidden = pd.concat(hidden_parts)

    train_input = pd.concat([non_validation_ratings, validation_visible])

    return train_input, validation_hidden

for fold in folds:
    train_input, validation_ground_truth = create_all_but_n_split(
        ratings=remaining_set,
        validation_users=fold["validation_users"],
        n=11,
        random_state=42
    )

    fold["train_input"] = train_input
    fold["validation_ground_truth"] = validation_ground_truth

    print(f"\nFold {fold['fold']}")
    print("Train input ratings:", len(train_input))
    print("Validation hidden ratings:", len(validation_ground_truth))

In [ ]:
for fold in folds:
    train_data = fold["train_input"] # train dataframe to train the recommender
    validation_data = fold["validation_ground_truth"] # validation dataframe to test the recommendations

In [ ]:
# Metrics

# ---------- Helper function ----------

def cosine_similarity(vec1, vec2):
    """
    Compute cosine similarity between two vectors.

    Parameters:
        vec1 (array-like): First vector
        vec2 (array-like): Second vector

    Returns:
        float: Cosine similarity
    """
    denom = np.linalg.norm(vec1) * np.linalg.norm(vec2)

    if denom == 0:
        return 0.0

    return np.dot(vec1, vec2) / denom

# ---------- Accuracy metrics ----------

def mae(y_true, y_pred):
    """
    Mean Absolute Error (MAE)

    Parameters:
        y_true (array-like): True ratings
        y_pred (array-like): Predicted ratings

    Returns:
        float: MAE value
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return np.mean(np.abs(y_true - y_pred))


def rmse(y_true, y_pred):
    """
    Root Mean Square Error (RMSE)

    Parameters:
        y_true (array-like): True ratings
        y_pred (array-like): Predicted ratings

    Returns:
        float: RMSE value
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    return np.sqrt(np.mean((y_true - y_pred) ** 2))



# ---------- Top-N relevance metrics ----------

def precision_at_n(recommendations_df, n=5, relevance_threshold=4.0):
    """
    Mean Precision@N across all users.

    Parameters:
        recommendations_df (DataFrame): Final recommendations.
            Must contain:
            - userId
            - movieId
            - rating
            - predicted_rating

        n (int): Number of top recommendations
        relevance_threshold (float): Minimum true rating to count as relevant

    Returns:
        float: Mean Precision@N
    """
    precision_scores = []

    for user_id, user_recs in recommendations_df.groupby("userId"):
        top_n = user_recs.sort_values("predicted_rating", ascending=False).head(n)

        if len(top_n) == 0:
            precision_scores.append(1.0)
        else:
            relevant_count = (top_n["rating"] >= relevance_threshold).sum()
            precision_scores.append(relevant_count / len(top_n))

    return np.mean(precision_scores)


def recall_at_n(recommendations_df, n=5, relevance_threshold=4.0):
    """
    Mean Recall@N across all users.

    Parameters:
        recommendations_df (DataFrame): Final recommendations.
            Must contain:
            - userId
            - movieId
            - rating
            - predicted_rating

        n (int): Number of top recommendations
        relevance_threshold (float): Minimum true rating to count as relevant

    Returns:
        float: Mean Recall@N
    """
    recall_scores = []

    for user_id, user_recs in recommendations_df.groupby("userId"):
        relevant_items = user_recs[user_recs["rating"] >= relevance_threshold]

        if len(relevant_items) == 0:
            recall_scores.append(1.0)
            continue

        top_n = user_recs.sort_values("predicted_rating", ascending=False).head(n)
        relevant_count = (top_n["rating"] >= relevance_threshold).sum()

        recall_scores.append(relevant_count / len(relevant_items))

    return np.mean(recall_scores)


# ---------- Beyond-accuracy metrics ----------

def novelty(recommendations_df, user_history, n=5):
    """
    Mean Novelty@N across all users.

    Novelty measures how unlikely it is that the user already knows
    the recommended items.

    This implementation follows two ideas:
    1. Recommended items should be new to the user:
       R_u ∩ I_u = empty set

    2. Less popular items are considered more novel:
       novelty_u = (1 / |R_u|) * sum(-log(popularity(i)))

    Parameters:
        recommendations_df (DataFrame): Final recommendations.
            Must contain:
            - userId
            - movieId
            - predicted_rating

        user_history (DataFrame): User interaction history.
            Must contain:
            - userId
            - movieId

        n (int): Number of top recommendations

    Returns:
        float: Mean Novelty@N across all users
    """
    novelty_scores = []

    # Count how often each movie appears in the user history
    item_counts = user_history["movieId"].value_counts()

    # Convert counts into popularity probabilities
    item_popularity = item_counts / item_counts.sum()

    # Store watched movies per user for fast lookup
    watched_by_user = (
        user_history
        .groupby("userId")["movieId"]
        .apply(set)
        .to_dict()
    )

    for user_id, user_recs in recommendations_df.groupby("userId"):

        # Select top-N recommendations
        top_n = (
            user_recs
            .sort_values("predicted_rating", ascending=False)
            .head(n)
        )

        if len(top_n) == 0:
            continue

        watched_items = watched_by_user.get(user_id, set())

        item_novelty_scores = []

        for movie_id in top_n["movieId"]:

            # If the user has already watched the item, it is not novel
            if movie_id in watched_items:
                item_novelty_scores.append(0.0)

            else:
                # Popularity-based novelty: -log(popularity(i))
                popularity = item_popularity.get(movie_id, 1 / len(item_popularity))
                item_novelty_scores.append(-np.log(popularity))

        novelty_scores.append(np.mean(item_novelty_scores))

    return np.mean(novelty_scores)


def diversity(recommendations_df, item_features, n=5):
    """
    Mean Diversity@N across all users.

    Diversity measures how different the recommended items are
    from each other using 1 - cosine similarity.

    Parameters:
        recommendations_df (DataFrame): Final recommendations.
            Must contain:
            - userId
            - movieId
            - predicted_rating

        item_features (dict): Mapping movieId -> feature vector

        n (int): Number of top recommendations

    Returns:
        float: Mean Diversity@N
    """
    diversity_scores = []

    for user_id, user_recs in recommendations_df.groupby("userId"):
        top_n = user_recs.sort_values("predicted_rating", ascending=False).head(n)
        items = top_n["movieId"].values

        if len(items) <= 1:
            diversity_scores.append(1.0)
            continue

        similarities = []

        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                if items[i] in item_features and items[j] in item_features:
                    vec_i = item_features[items[i]]
                    vec_j = item_features[items[j]]

                    similarities.append(cosine_similarity(vec_i, vec_j))

        if len(similarities) == 0:
            diversity_scores.append(1.0)
        else:
            intra_list_similarity = np.mean(similarities)
            diversity_scores.append(1 - intra_list_similarity)

    return np.mean(diversity_scores)


def serendipity(
    recommendations_df,
    user_history,
    item_features,
    n=5,
    relevance_threshold=4.0,
    similarity_threshold=0.5
):
    """
    Mean Serendipity@N across all users.

    Serendipity measures the proportion of recommended items that are:
    - relevant: true rating >= relevance_threshold
    - unexpected: average similarity to user history < similarity_threshold

    Parameters:
        recommendations_df (DataFrame): Final recommendations.
            Must contain:
            - userId
            - movieId
            - rating
            - predicted_rating

        user_history (DataFrame): User interaction history.
            Must contain:
            - userId
            - movieId

        item_features (dict): Mapping movieId -> feature vector

        n (int): Number of top recommendations
        relevance_threshold (float): Minimum true rating to count as relevant
        similarity_threshold (float): Maximum average similarity to count as unexpected

    Returns:
        float: Mean Serendipity@N
    """
    serendipity_scores = []

    for user_id, user_recs in recommendations_df.groupby("userId"):
        top_n = user_recs.sort_values("predicted_rating", ascending=False).head(n)

        if len(top_n) == 0:
            serendipity_scores.append(1.0)
            continue

        history_items = user_history[user_history["userId"] == user_id]["movieId"].values
        serendipitous_count = 0

        for _, row in top_n.iterrows():
            movie_id = row["movieId"]
            true_rating = row["rating"]

            if true_rating < relevance_threshold:
                continue

            if movie_id not in item_features:
                continue

            similarities = []

            for hist_item in history_items:
                if hist_item in item_features:
                    sim = cosine_similarity(
                        item_features[movie_id],
                        item_features[hist_item]
                    )
                    similarities.append(sim)

            if len(similarities) == 0:
                serendipitous_count += 1
            elif np.mean(similarities) < similarity_threshold:
                serendipitous_count += 1

        serendipity_scores.append(serendipitous_count / len(top_n))

    return np.mean(serendipity_scores)

### Exercise 4 - Optimize hyperparameters of baseline RS (6 points)
Optimize the hyperparameters $\beta_u$ and $\beta_i$ for the baseline RS from exercise 2 based on the RMSE metric. To save computation time find a reasonable maximum value for the betas. Explain your approach and your solution.
Plot the MAE, RMSE, Precision@N, Recall@N as functions of the betas.

Which metric would you use for hyperparameter tuning? Explain your decision.

In [ ]:
# ---------- Baseline hyperparameter optimization ----------

def predict_baseline_for_validation(train_data, validation_data, beta_u, beta_i):
    mu = train_data["rating"].mean()

    b_u = compute_user_bias(train_data, mu, beta_u=beta_u)
    b_i = compute_item_bias(train_data, mu, b_u, beta_i=beta_i)

    predictions_df = validation_data.copy()

    predictions_df["b_u"] = predictions_df["userId"].map(b_u).fillna(0)
    predictions_df["b_i"] = predictions_df["movieId"].map(b_i).fillna(0)

    predictions_df["predicted_rating"] = (
        mu + predictions_df["b_u"] + predictions_df["b_i"]
    )

    return predictions_df


def evaluate_baseline_cv(folds, beta_u, beta_i, n=5, relevance_threshold=4.0):
    """
    Evaluate one beta_u / beta_i combination across all folds.
    """
    mae_scores = []
    rmse_scores = []
    precision_scores = []
    recall_scores = []

    for fold in folds:
        train_data = fold["train_input"]
        validation_data = fold["validation_ground_truth"]

        predictions_df = predict_baseline_for_validation(
            train_data=train_data,
            validation_data=validation_data,
            beta_u=beta_u,
            beta_i=beta_i
        )

        y_true = predictions_df["rating"].values
        y_pred = predictions_df["predicted_rating"].values

        mae_scores.append(mae(y_true, y_pred))
        rmse_scores.append(rmse(y_true, y_pred))

        precision_scores.append(
            precision_at_n(
                predictions_df,
                n=n,
                relevance_threshold=relevance_threshold
            )
        )

        recall_scores.append(
            recall_at_n(
                predictions_df,
                n=n,
                relevance_threshold=relevance_threshold
            )
        )

    return {
        "beta_u": beta_u,
        "beta_i": beta_i,
        "MAE": np.mean(mae_scores),
        "RMSE": np.mean(rmse_scores),
        "Precision@N": np.mean(precision_scores),
        "Recall@N": np.mean(recall_scores)
    }

In [ ]:
beta_values = [0, 5, 10, 100, 200]

results = []

for beta_u in beta_values:
    for beta_i in beta_values:
        result = evaluate_baseline_cv(
            folds=folds,
            beta_u=beta_u,
            beta_i=beta_i,
            n=5,
            relevance_threshold=4.0
        )
        results.append(result)

results_df = pd.DataFrame(results)

results_df.sort_values("RMSE").head(10)

# Best results
best_result = results_df.loc[results_df["RMSE"].idxmin()]

print("Best beta_u:", best_result["beta_u"])
print("Best beta_i:", best_result["beta_i"])
print("Best RMSE:", best_result["RMSE"])
print("MAE:", best_result["MAE"])
print("Precision@N:", best_result["Precision@N"])
print("Recall@N:", best_result["Recall@N"])

In [ ]:
def plot_metric_heatmap(results_df, metric):
    """
    One metric as a function of beta_u and beta_i.
    """
    pivot = results_df.pivot(
        index="beta_u",
        columns="beta_i",
        values=metric
    )

    plt.figure(figsize=(8, 6))
    plt.imshow(pivot.values, aspect="auto", origin="lower")

    plt.colorbar(label=metric)
    plt.xticks(range(len(pivot.columns)), pivot.columns)
    plt.yticks(range(len(pivot.index)), pivot.index)

    plt.xlabel("beta_i")
    plt.ylabel("beta_u")
    plt.title(f"{metric} for different beta values")

    plt.show()


plot_metric_heatmap(results_df, "MAE")
plot_metric_heatmap(results_df, "RMSE")
plot_metric_heatmap(results_df, "Precision@N")
plot_metric_heatmap(results_df, "Recall@N")

**Hyperparameter Optimization of the Baseline Recommender**

To optimize the hyperparameters $\beta_u$ and $\beta_i$, I performed a **grid search** over a range of values [0, 5, 10, 50, 100, 200]. This range was chosen to balance computational cost and coverage of meaningful regularization strengths. Larger values were not considered, as increasing $\beta$ increasingly shrinks user and item biases toward zero, making the model approach the global mean predictor.

**Results**

The optimal hyperparameters based on RMSE are:

* $\beta_u = 5$
* $\beta_i = 5$
* **RMSE = 0.8951**

Additional metrics at this configuration:

* MAE = 0.6876
* Precision@N = 0.7393
* Recall@N = 0.1928

**Interpretation of Heatmaps**

The heatmaps show that:

* **Low to moderate beta values (around 5–10)** yield the best performance for RMSE and MAE
* Very high beta values (100–200) significantly degrade performance, as they over-regularize the model
* Precision@N slightly increases with higher $\beta_i$ but this comes at the cost of worse RMSE
* Recall@N varies only slightly and does not strongly influence the optimal choice

Overall, both error-based metrics (RMSE, MAE) clearly indicate a minimum around $\beta_u$ $\beta_i$ = (5,5).


**Choice of Metric for Hyperparameter Tuning**

I would use **RMSE** for hyperparameter tuning.

This is because:

* The baseline recommender is a **rating prediction model**, and RMSE directly measures prediction accuracy
* RMSE penalizes larger errors more strongly than MAE, making it more sensitive and suitable for optimization
* Precision@N and Recall@N depend on a relevance threshold and focus on ranking quality rather than prediction accuracy


## Final Conclusion

The optimal configuration is:

$$ \beta_u = 5, \quad \beta_i = 5 $$

This provides the best trade-off between model flexibility and regularization, resulting in the lowest prediction error while maintaining stable recommendation quality.

### Exercise 5 - Collaborative filtering; item-based and user-based (16 points)
In this exersise we will build 8 different collaborative-filtering RS based on nearest neighbour technique, both in terms of item and user. 

Implement:
1. a RS based on the $K$ most similar items (K nearest neighbours). Similarity shall be calculated based on *cosine similarity*. 
2. a RS based on the $K$ most similar items (K nearest neighbours). Similarity shall be calculated based on *Pearson Correlation Coefficienct*. 
3. a RS based on the $K$ most similar users (K nearest neighbours). Similarity shall be calculated based on *cosine similarity*. 
4. a RS based on the $K$ most similar users (K nearest neighbours). Similarity shall be calculated based on *Pearson Correlation Coefficienct*. 

Make for each of the models above 2 variants. One with user-wise zero centering and one without.

- User-wise zero centering is a preprocessing step that transforms each user’s rating vector so that its mean value is zero. This step is inherently part of Pearson correlation, but when applied to the entire dataset beforehand, it serves as a global normalization and influences all similarity computations.

Each should have a default $K$ of 20.

Explain how you handle NaN values in the user rating matrix when computing similarities? What other preparations are useful such as normalization and mean centering?

- Missing values (NaN) in the user–item matrix are not replaced with zero. Instead, similarities are computed only on the subset of users or items that have co-rated the same entities. This avoids introducing artificial information, since a missing rating does not imply a negative preference.

For both cosine similarity and Pearson correlation, only overlapping ratings are considered when computing similarity.

Describe the two similarity metrics.

- **Cosine similarity** compares the angle between two rating vectors. It measures how similar the rating patterns are in direction, independent of their magnitude. In collaborative filtering, it is computed only on co-rated entries without modifying the ratings.

- **Pearson correlation** reduces rating biases by centering ratings around their mean. Instead of comparing absolute ratings, it measures the similarity of rating deviations from the average. This allows similar preference patterns to be identified even when users or items have different baseline rating levels.

Show the top 20 recommended items for user ids 3, 5 and 7.

In [ ]:
# --- similarity metrics ---

def pearson_correlation(vec1, vec2):
    """
    Pearson correlation between two vectors.

    Parameters:
        vec1 (array-like): First vector (may contain NaN)
        vec2 (array-like): Second vector (may contain NaN)

    Returns:
        float: Pearson correlation coefficient
    """
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    # only co-rated entries (ignore NaNs)
    mask = ~np.isnan(vec1) & ~np.isnan(vec2)

    if np.sum(mask) < 2:
        return 0.0

    v1 = vec1[mask]
    v2 = vec2[mask]

    # mean-centering
    v1_centered = v1 - np.mean(v1)
    v2_centered = v2 - np.mean(v2)

    denom = np.linalg.norm(v1_centered) * np.linalg.norm(v2_centered)

    if denom == 0:
        return 0.0

    return np.dot(v1_centered, v2_centered) / denom

def cosine_similarity(vec1, vec2):
    """
    Cosine similarity between two vectors.

    Parameters:
        vec1 (array-like): First vector (may contain NaN)
        vec2 (array-like): Second vector (may contain NaN)

    Returns:
        float: Cosine similarity
    """
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    # Only co-rated entries (ignore NaNs)
    mask = ~np.isnan(vec1) & ~np.isnan(vec2)

    if np.sum(mask) < 1:
        return 0.0

    v1 = vec1[mask]
    v2 = vec2[mask]

    denom = np.linalg.norm(v1) * np.linalg.norm(v2)

    if denom == 0:
        return 0.0

    return np.dot(v1, v2) / denom


# --- user-wise zero-centering ---
def user_mean_centering(ratings):
    user_means = ratings.groupby("userId")["rating"].transform("mean")
    ratings_centered = ratings.copy()
    ratings_centered["rating"] = ratings["rating"] - user_means
    return ratings_centered

# --- item and user matrixes ---
def create_user_item_matrix(ratings):
    """
    Create user-item rating matrix.
    Rows:
        users
    Columns:
        movies
    Values:
        ratings
    """
    return ratings.pivot_table(index="userId",columns="movieId",values="rating")

# --- similarity matrixes ---
def compute_item_similarity_matrix(user_item_matrix, similarity_func):
    movie_ids = user_item_matrix.columns

    similarity_matrix = pd.DataFrame(
        index=movie_ids,
        columns=movie_ids,
        dtype=float
    )

    for movie_i in movie_ids:
        for movie_j in movie_ids:
            if movie_i == movie_j:
                similarity_matrix.loc[movie_i, movie_j] = 1.0
            else:
                similarity_matrix.loc[movie_i, movie_j] = similarity_func(
                    user_item_matrix[movie_i].values,
                    user_item_matrix[movie_j].values
                )

    return similarity_matrix


def compute_user_similarity_matrix(user_item_matrix, similarity_func):
    user_ids = user_item_matrix.index

    similarity_matrix = pd.DataFrame(
        index=user_ids,
        columns=user_ids,
        dtype=float
    )

    for user_i in user_ids:
        for user_j in user_ids:
            if user_i == user_j:
                similarity_matrix.loc[user_i, user_j] = 1.0
            else:
                similarity_matrix.loc[user_i, user_j] = similarity_func(
                    user_item_matrix.loc[user_i].values,
                    user_item_matrix.loc[user_j].values
                )

    return similarity_matrix


# --- item-based prediction ---
def predict_item_based(user_id, movie_id, user_item_matrix, item_similarity_matrix, k=20):
    user_ratings = user_item_matrix.loc[user_id]
    rated_movies = user_ratings.dropna()

    if len(rated_movies) == 0:
        return np.nan

    similarities = item_similarity_matrix.loc[movie_id, rated_movies.index]

    similarities = similarities[similarities > 0]

    if len(similarities) == 0:
        return np.nan

    top_k_similarities = similarities.sort_values(ascending=False).head(k)
    top_k_ratings = rated_movies.loc[top_k_similarities.index]

    prediction = np.dot(top_k_similarities, top_k_ratings) / np.sum(np.abs(top_k_similarities))

    return prediction

# --- user-based prediction ---
def predict_user_based(user_id, movie_id, user_item_matrix, user_similarity_matrix, k=20):
    movie_ratings = user_item_matrix[movie_id]
    users_who_rated = movie_ratings.dropna()

    if len(users_who_rated) == 0:
        return np.nan

    similarities = user_similarity_matrix.loc[user_id, users_who_rated.index]

    similarities = similarities[similarities > 0]

    if len(similarities) == 0:
        return np.nan

    top_k_similarities = similarities.sort_values(ascending=False).head(k)
    top_k_ratings = users_who_rated.loc[top_k_similarities.index]

    prediction = np.dot(top_k_similarities, top_k_ratings) / np.sum(np.abs(top_k_similarities))

    return prediction


# THE KNN RECOMMENDER
def recommend_knn(
    user_id,
    user_item_matrix,
    similarity_matrix,
    model_type="item",
    k=20,
    n=20,
    candidate_movies=None
):
    user_ratings = user_item_matrix.loc[user_id]

    if candidate_movies is None:
        all_movies = user_ratings[user_ratings.isna()].index
    else:
        all_movies = [
            movie_id for movie_id in candidate_movies
            if movie_id in user_item_matrix.columns
        ]

    recommendations = []

    for movie_id in all_movies:

        if model_type == "item":
            prediction = predict_item_based(
                user_id,
                movie_id,
                user_item_matrix,
                similarity_matrix,
                k
            )

        elif model_type == "user":
            prediction = predict_user_based(
                user_id,
                movie_id,
                user_item_matrix,
                similarity_matrix,
                k
            )

        else:
            raise ValueError("model_type must be 'item' or 'user'")

        if not np.isnan(prediction):
            recommendations.append({
                "userId": user_id,
                "movieId": movie_id,
                "predicted_rating": prediction
            })

    recommendations_df = pd.DataFrame(recommendations)

    if len(recommendations_df) == 0:
        return recommendations_df

    recommendations_df["predicted_rating"] = (
        recommendations_df["predicted_rating"]
        .clip(0.5, 5.0)
        .round(3)
    )

    return recommendations_df.sort_values(
        by="predicted_rating",
        ascending=False
    ).head(n)

In [ ]:
# --- prepare folds for KNN ---

for fold in folds:
    print(f"\nPreparing fold {fold['fold']}...")

    train_input, validation_ground_truth = create_all_but_n_split(
        ratings=remaining_set,
        validation_users=fold["validation_users"],
        n=11,
        random_state=42
    )

    fold["train_input"] = train_input
    fold["validation_ground_truth"] = validation_ground_truth

    # non-centered user-item matrix
    fold["matrix"] = create_user_item_matrix(train_input)

    # centered user-item matrix
    train_input_centered = user_mean_centering(train_input)
    fold["train_input_centered"] = train_input_centered
    fold["matrix_centered"] = create_user_item_matrix(train_input_centered)

    # user means for centered predictions
    fold["user_means"] = train_input.groupby("userId")["rating"].mean()

    print("Train input ratings:", len(train_input))
    print("Validation hidden ratings:", len(validation_ground_truth))
    print("Non-centered matrix shape:", fold["matrix"].shape)
    print("Centered matrix shape:", fold["matrix_centered"].shape)
    print(f"Fold {fold['fold']} prepared.")

In [ ]:
# --- helper functions for fast similarity (per user only) --- (necessary otherwise process takes way too long)

def compute_item_similarity_for_user(user_id, user_item_matrix, similarity_func):
    user_ratings = user_item_matrix.loc[user_id]
    rated_movies = user_ratings.dropna().index
    all_movies = user_ratings[user_ratings.isna()].index

    sim_matrix = pd.DataFrame(index=all_movies, columns=rated_movies, dtype=float)

    for movie_i in all_movies:
        for movie_j in rated_movies:
            if movie_i == movie_j:
                sim_matrix.loc[movie_i, movie_j] = 1.0
            else:
                sim_matrix.loc[movie_i, movie_j] = similarity_func(
                    user_item_matrix[movie_i].values,
                    user_item_matrix[movie_j].values
                )

    return sim_matrix


def compute_user_similarity_for_user(user_id, user_item_matrix, similarity_func):
    all_users = user_item_matrix.index

    sim_matrix = pd.DataFrame(index=[user_id], columns=all_users, dtype=float)

    for other_user in all_users:
        if user_id == other_user:
            sim_matrix.loc[user_id, other_user] = 1.0
        else:
            sim_matrix.loc[user_id, other_user] = similarity_func(
                user_item_matrix.loc[user_id].values,
                user_item_matrix.loc[other_user].values
            )

    return sim_matrix

In [ ]:
# --- 8 knn models ---

models = [
    ("item_cosine", "item", cosine_similarity, False),
    ("item_pearson", "item", pearson_correlation, False),
    ("user_cosine", "user", cosine_similarity, False),
    ("user_pearson", "user", pearson_correlation, False),

    ("item_cosine_centered", "item", cosine_similarity, True),
    ("item_pearson_centered", "item", pearson_correlation, True),
    ("user_cosine_centered", "user", cosine_similarity, True),
    ("user_pearson_centered", "user", pearson_correlation, True),
]

In [ ]:
# --- all 8 KNN models for users 3, 5, 7 using first fold --- (also necessary otherwise process takes way too long)

selected_fold = folds[0]
target_users = [3, 5, 7]


for model_name, model_type, similarity_func, is_centered in models:

    print(f"\n===== {model_name} =====")

    current_matrix = selected_fold["matrix_centered"] if is_centered else selected_fold["matrix"]

    for user_id in target_users:

        if model_type == "item":
            sim_matrix = compute_item_similarity_for_user(
                user_id,
                current_matrix,
                similarity_func
            )
        else:
            sim_matrix = compute_user_similarity_for_user(
                user_id,
                current_matrix,
                similarity_func
            )

        recs = recommend_knn(
            user_id=user_id,
            user_item_matrix=current_matrix,
            similarity_matrix=sim_matrix,
            model_type=model_type,
            k=20,
            n=20
        )

        if is_centered:
            recs["predicted_rating"] = (
                recs["predicted_rating"] + selected_fold["user_means"].loc[user_id]
            )

        print(f"\nUser {user_id}")
        print(recs)

### Report: Implementation and Runtime Issues in the KNN Collaborative Filtering Models

During the implementation of the KNN-based collaborative filtering recommenders, several adjustments were necessary before the models could be applied correctly to users `3`, `5`, and `7`.

At first, the dataset was split by taking the first 20% of sorted users as the test set. This caused a problem because users `3`, `5` and `7` were placed inside the held-out test set. Since the test set is not allowed to be used until Exercise 10, recommendations for these users could not be generated from the available training data. To solve this, the splitting function was changed so that the test users are selected randomly, while users `3`, `5`, and `7` are explicitly kept in the remaining training data. This made it possible to test the recommenders on the required users without using the forbidden test set.

Another major issue was runtime. The first implementation computed full item-item and user-user similarity matrices. This was way too slow, especially for item-based KNN, because the dataset contains several thousand movies. Computing a full item similarity matrix means comparing every movie with every other movie, which results in millions of similarity calculations. This made the code impractical for testing all eight models.

To improve performance, the computation was moved outside the recommender where possible, and additional helper functions were introduced. Instead of computing full similarity matrices, the optimized version computes only the similarities needed for the target users. For user-based KNN, this means comparing the target user with all other users. For item-based KNN, this means comparing candidate movies only with movies that the target user has already rated. This keeps the recommendation logic the same but avoids many unnecessary calculations.

A further change was added to avoid recommending movies that the user had already rated. This also reduced runtime and made the recommendations more meaningful. Since a recommender system should normally suggest unknown items, filtering out already-rated movies is a reasonable constraint.

The final implementation evaluated all eight required model variants:

1. item-based cosine
2. item-based Pearson
3. user-based cosine
4. user-based Pearson
5. item-based cosine with user-wise zero-centering
6. item-based Pearson with user-wise zero-centering
7. user-based cosine with user-wise zero-centering
8. user-based Pearson with user-wise zero-centering

After the optimizations, the full run still took around 35 minutes. I do not have an especially strong laptop, but even is stronger hardware I expect the cell to take about 25 minutes. The item-based models were the slowest, while the user-based models were almost instant.

The results show that many models produced very high predicted ratings, often close to or equal to `5.0`. This was especially visible in the non-centered user-based models and the Pearson-based models. This suggests that the KNN models are strongly influenced by highly similar neighbours with high ratings. The centered user-based models produced more varied results, but some predictions exceeded the MovieLens maximum rating of `5.0`, I fixed that by clipping predictions to the valid rating range and wrote a line to toggle to the movielens rating scale, that only allows `0.5`, `1.0` and so on.

Overall, the final implementation successfully generated top-20 recommendations for users `3`, `5`, and `7` for all eight KNN variants. However, the process showed that naive KNN implementations can be computationally expensive, especially for item-based collaborative filtering. Efficient similarity computation and candidate filtering were necessary to make the models usable.



### Exercise 6 - Optimize hyperparameter $K$ (6 points)
Optimize the hyperparameter $K$ for all RS from the prior exercise optimizing for minimal RMSE. 
For each RS plot RMSE, Precision@N and Recall@N as a function of $K$. 

Compare the results of these four RS on the 3 example users. Do the results match your expectation? Describe.

In [ ]:
# I tried

### Exercise 7 - Model-based RS: SVD (10 points)
In this exercise we will use the unsupervised method *singular value decomposition (SVD)* from the python package *surprise* (https://surpriselib.com, documentation https://surprise.readthedocs.io/en/stable/matrix_factorization.html). SVD can compress much of the information of a matrix in few components.  

a)Run the SVD RS and show the results on the three example users from exercise 2. Explain how this algorithm works.

Note: A very good general introduction to SVD is this youtube video series starting with https://www.youtube.com/watch?v=gXbThCXjZFM&t=337s . See *Collaborative filtering recommender systems* by Ekstrand et al. *Mining of massive datasets* by Leskovec, Kapitel 11 (2020) and, *Recommender systems: The textbook*, by Aggarwal, chapter 3

b) We explore now what latent factors SVD has learned. Generate an interactive 2D UMAP plot of all latent movie factors. Make versions of this plot with fixed values for *n_neighbours* and *min_dist* each time coloring the points by different parameters: year, genre, numbers of genres, average rating and number of ratings to explore the resulting plot. Can you see any patterns?

Background: UMAP is a method for dimensionality reduction. Dimensionality reduction is usefull to respresent high dimensional data sets in less dimensions with goal to allow interpretable 2D/3D visualization. See for the documentation of the python package https://umap-learn.readthedocs.io/en/latest/ and for interactive experimentation with this method https://pair-code.github.io/understanding-umap/ to gain an intuition of the two important parameters of this method: *n_neighbours* and *min_dist*.

**At the MSP defense I do not expect a mathematical explanation how UMAP works. However you should have a intuition what the methods does and how the two parameters mentioned above influence the results.**


### Exercise 8 - Optimize hyperparameter $k$ or `n_factors` (4 points)
Optimize the hyperparameter, representing the number of greatest SVD components used for the truncated reconstruction of the user item matrix, to minimize RMSE.
Plot RMSE, Precision@N and Recall@N as a function of this hyperparameter. Finally output all performance metrics from exercise 3 for the optimal $k$ value.

### Exercise 9 - Everything goes (30 points)
In this exercise you can explore different methods of RS. You are not limited what methods you apply. You can try to improve the methods from the earlier exercises by modifiying them or generating ensemble or hybrid RS. Also you could train deep neural networks, use NLP methods, use the available links to imdb available in the dataset to further enrich the dataset or find an obscure method by someone else on Github. 
Document what your inspirations and sources are and describe the method conceptually. 

**Build and optimize in total *three* different methods. The last one has the additional requirement that it should increase the diversity of the recommendations in order to minimize filter bubbles.**

**Important: If you use the work of someone else you must be able to explain the method conceptually during the defense MSP.** 

Output the performance metrics of exercise 3. 

### Exercise 10 - Compare all RS that you build in this challenge (8 points)
a) Compile a table with the performance metrics of exercise 3 for all RS from this MC (Make sure to include the baseline RS and random RS) on the test set defined in exercise 3. Also generate comparative plots. Discuss.

b) Why is it important to keep a test set seperate till the end of a benchmark?

**Read the Guidelines for Implementation and Submission one more time.**